In [1]:
import os
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
    mean_squared_error
)

print("SwinIR Restoration Experiment")
print("PyTorch:", torch.__version__)

SwinIR Restoration Experiment
PyTorch: 2.5.1+cu121


In [2]:
SEED = 42


random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Seed:", SEED)
print("Device:", device)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Seed: 42
Device: cuda
GPU: Quadro GV100


In [3]:
GT_PATH = "../data/train/GT"

NOISY_PATH = "../data/train/NoisyLR"


files = sorted(
    [
        f
        for f in os.listdir(GT_PATH)
        if f.endswith(".npy")
    ]
)


print(
    "Total image pairs:",
    len(files)
)

Total image pairs: 3200


In [4]:
np.random.seed(SEED)


indices = np.random.permutation(
    len(files)
)



train_end = int(
    0.80 * len(files)
)


val_end = int(
    0.90 * len(files)
)



train_files = [
    files[i]
    for i in indices[:train_end]
]


val_files = [
    files[i]
    for i in indices[
        train_end:val_end
    ]
]


test_files = [
    files[i]
    for i in indices[
        val_end:
    ]
]


print(
    "Train:",
    len(train_files)
)


print(
    "Validation:",
    len(val_files)
)


print(
    "Test:",
    len(test_files)
)

Train: 2560
Validation: 320
Test: 320


In [5]:
class KLADataset(Dataset):

    def __init__(
        self,
        gt_path,
        noisy_path,
        files
    ):

        self.gt_path = gt_path
        self.noisy_path = noisy_path
        self.files = files



    def __len__(self):

        return len(self.files)



    def __getitem__(
        self,
        idx
    ):

        filename = self.files[idx]


        lr = np.load(
            os.path.join(
                self.noisy_path,
                filename
            )
        ).astype(
            np.float32
        )



        gt = np.load(
            os.path.join(
                self.gt_path,
                filename
            )
        ).astype(
            np.float32
        )



        lr = torch.from_numpy(
            lr
        ).unsqueeze(0)



        gt = torch.from_numpy(
            gt
        ).unsqueeze(0)



        return lr,gt

In [6]:
train_dataset = KLADataset(
    GT_PATH,
    NOISY_PATH,
    train_files
)


val_dataset = KLADataset(
    GT_PATH,
    NOISY_PATH,
    val_files
)


test_dataset = KLADataset(
    GT_PATH,
    NOISY_PATH,
    test_files
)



print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)

2560 320 320


In [7]:
BATCH_SIZE = 8


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)



print("DataLoaders ready")

DataLoaders ready


In [8]:
lr,gt = next(
    iter(train_loader)
)


print(
    "LR shape:",
    lr.shape
)


print(
    "GT shape:",
    gt.shape
)

LR shape: torch.Size([8, 1, 128, 128])
GT shape: torch.Size([8, 1, 256, 256])


In [9]:
MODEL_DIR = "../models/swinir_sr"

RESULT_DIR = "../results/swinir_sr"


os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "swinir_best.pth"
)


print(
    BEST_MODEL_PATH
)

../models/swinir_sr/swinir_best.pth


In [10]:
class WindowAttentionBlock(nn.Module):

    def __init__(
        self,
        channels,
        window_size=8
    ):
        super().__init__()

        self.window_size = window_size

        self.norm = nn.LayerNorm(
            channels
        )


        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=8,
            batch_first=True
        )


    def forward(self,x):

        b,c,h,w = x.shape


        ws = self.window_size


        pad_h = (
            ws - h % ws
        ) % ws

        pad_w = (
            ws - w % ws
        ) % ws


        if pad_h or pad_w:

            x = F.pad(
                x,
                (
                    0,
                    pad_w,
                    0,
                    pad_h
                )
            )


        _,_,hp,wp = x.shape


        # window partition

        x = x.permute(
            0,
            2,
            3,
            1
        )


        windows = x.reshape(
            b,
            hp//ws,
            ws,
            wp//ws,
            ws,
            c
        )


        windows = windows.permute(
            0,
            1,
            3,
            2,
            4,
            5
        )


        windows = windows.reshape(
            -1,
            ws*ws,
            c
        )


        residual = windows


        windows = self.norm(
            windows
        )


        attn,_ = self.attention(
            windows,
            windows,
            windows
        )


        windows = residual + attn



        # merge windows

        windows = windows.reshape(
            b,
            hp//ws,
            wp//ws,
            ws,
            ws,
            c
        )


        windows = windows.permute(
            0,
            1,
            3,
            2,
            4,
            5
        )


        out = windows.reshape(
            b,
            hp,
            wp,
            c
        )


        out = out.permute(
            0,
            3,
            1,
            2
        )


        return out[:,:,:h,:w]

In [11]:
class SwinIR_SR(nn.Module):

    def __init__(
        self,
        channels=32,
        blocks=4,
        heads=4
    ):

        super().__init__()


        self.head = nn.Conv2d(
            1,
            channels,
            kernel_size=3,
            padding=1
        )


        self.blocks = nn.ModuleList(
            [
                WindowAttentionBlock(
                    channels,
                    heads
                )
                for _ in range(blocks)
            ]
        )


        self.body_conv = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )


        self.up = nn.Sequential(

            nn.Conv2d(
                channels,
                channels*4,
                3,
                padding=1
            ),

            nn.PixelShuffle(
                2
            ),


            nn.Conv2d(
                channels,
                1,
                3,
                padding=1
            )
        )



    def forward(self,x):

        x = self.head(x)

        skip = x


        for blk in self.blocks:

            x = blk(x)


        x = self.body_conv(x)

        x = x + skip


        x = self.up(x)


        return x

In [12]:
import gc
import torch


gc.collect()

torch.cuda.empty_cache()



model = SwinIR_SR(
    channels=32,
    blocks=4,
    heads=4
)


model = model.to(device)



params = sum(
    p.numel()
    for p in model.parameters()
)


print(
    "Parameters:",
    params
)

Parameters: 64001


In [13]:
x = torch.randn(
    1,
    1,
    128,
    128
).to(device)



with torch.no_grad():

    y = model(x)



print(
    "Input:",
    x.shape
)


print(
    "Output:",
    y.shape
)

Input: torch.Size([1, 1, 128, 128])
Output: torch.Size([1, 1, 256, 256])


In [14]:
scaler = torch.cuda.amp.GradScaler()


print(
    "AMP enabled"
)

AMP enabled


/tmp/ipykernel_3752802/3214020012.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [15]:
def differentiable_ssim(
    img1,
    img2,
    window_size=11
):

    channels = img1.shape[1]


    window = torch.ones(
        channels,
        1,
        window_size,
        window_size,
        device=img1.device
    ) / (window_size**2)



    mu1 = F.conv2d(
        img1,
        window,
        padding=window_size//2,
        groups=channels
    )


    mu2 = F.conv2d(
        img2,
        window,
        padding=window_size//2,
        groups=channels
    )



    sigma1 = (
        F.conv2d(
            img1*img1,
            window,
            padding=window_size//2,
            groups=channels
        )
        -
        mu1**2
    )



    sigma2 = (
        F.conv2d(
            img2*img2,
            window,
            padding=window_size//2,
            groups=channels
        )
        -
        mu2**2
    )



    sigma12 = (
        F.conv2d(
            img1*img2,
            window,
            padding=window_size//2,
            groups=channels
        )
        -
        mu1*mu2
    )



    C1 = 0.01**2
    C2 = 0.03**2



    ssim = (

        ((2*mu1*mu2)+C1)
        *
        ((2*sigma12)+C2)

    ) / (

        (mu1**2 + mu2**2 + C1)
        *
        (sigma1 + sigma2 + C2)

        + 1e-8
    )


    return ssim.mean()

In [16]:
def swinir_loss(
    pred,
    target
):

    l1 = F.l1_loss(
        pred,
        target
    )


    ssim_value = differentiable_ssim(
        pred,
        target
    )


    loss = (
        l1
        +
        0.1*(1-ssim_value)
    )


    return loss

In [17]:
EPOCHS = 30


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)



print(
    "Optimizer ready"
)

Optimizer ready


In [18]:
best_psnr = 0



for epoch in range(EPOCHS):


    model.train()

    total_loss = 0



    for lr,gt in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    ):


        lr = lr.to(device)
        gt = gt.to(device)



        optimizer.zero_grad()



        with torch.cuda.amp.autocast():


            output = model(
                lr
            )


            loss = swinir_loss(
                output,
                gt
            )



        scaler.scale(
            loss
        ).backward()



        scaler.step(
            optimizer
        )


        scaler.update()



        total_loss += loss.item()



    scheduler.step()



    torch.cuda.empty_cache()



    print(
        "Epoch:",
        epoch+1,
        "Loss:",
        total_loss/len(train_loader)
    )

Epoch 1/30:   0%|          | 0/320 [00:00<?, ?it/s]/tmp/ipykernel_3752802/3473809498.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30: 100%|██████████| 320/320 [00:13<00:00, 22.99it/s]


Epoch: 1 Loss: 0.09352018012432381


Epoch 2/30: 100%|██████████| 320/320 [00:12<00:00, 24.69it/s]


Epoch: 2 Loss: 0.07374974959529937


Epoch 3/30: 100%|██████████| 320/320 [00:12<00:00, 25.05it/s]


...

Epoch: 28 Loss: 0.05686266031116247


Epoch 29/30: 100%|██████████| 320/320 [00:12<00:00, 24.79it/s]


Epoch: 29 Loss: 0.05683624900411814


Epoch 30/30: 100%|██████████| 320/320 [00:12<00:00, 24.64it/s]

Epoch: 30 Loss: 0.05683252232847735


In [19]:
MODEL_DIR="./models/swinir_sr"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


torch.save(
    {
        "model":
            model.state_dict(),

        "epoch":
            EPOCHS
    },

    MODEL_DIR+"/swinir_best.pth"
)


print(
    "SwinIR saved"
)

SwinIR saved


In [20]:
checkpoint=torch.load(
    "./models/swinir_sr/swinir_best.pth",
    map_location=device
)


model.load_state_dict(
    checkpoint["model"]
)


model.eval()


print(
    "Loaded SwinIR"
)

Loaded SwinIR


/tmp/ipykernel_3752802/1779626963.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint=torch.load(


In [21]:
with torch.no_grad():

    lr,gt = next(
        iter(test_loader)
    )


    lr = lr.to(device)


    restored = model(
        lr
    )


print(
    restored.shape
)

torch.Size([1, 1, 256, 256])


In [22]:
import time
import pandas as pd
import numpy as np

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)


model.eval()


results = []


with torch.no_grad():

    for idx,(lr,gt) in enumerate(
        tqdm(test_loader)
    ):


        lr = lr.to(device)


        start = time.time()


        pred = model(
            lr
        )


        torch.cuda.synchronize()


        inference_time = (
            time.time()-start
        )*1000



        pred = torch.clamp(
            pred,
            0,
            1
        )



        pred_np = (
            pred.cpu()
            .numpy()[0,0]
        )


        gt_np = (
            gt.numpy()[0,0]
        )



        psnr = peak_signal_noise_ratio(
            gt_np,
            pred_np,
            data_range=1
        )


        ssim = structural_similarity(
            gt_np,
            pred_np,
            data_range=1
        )


        mse = np.mean(
            (gt_np-pred_np)**2
        )



        results.append(
            {
                "Filename":
                    test_dataset.files[idx],

                "PSNR_dB":
                    psnr,

                "SSIM":
                    ssim,

                "MSE":
                    mse,

                "Inference_ms":
                    inference_time
            }
        )



swinir_results = pd.DataFrame(
    results
)


swinir_results.head()

100%|██████████| 320/320 [00:02<00:00, 111.19it/s]


,Filename,PSNR_dB,SSIM,MSE,Inference_ms
0,001218.npy,27.022462,0.759795,0.001985,7.003307
1,001158.npy,27.867224,0.514632,0.001634,3.353834
2,001415.npy,20.888544,0.694626,0.008150,2.831221
3,002896.npy,28.419508,0.780723,0.001439,2.821207
4,001527.npy,35.173669,0.893369,0.000304,2.844810


In [23]:
print(
"========== SWINIR FINAL RESULTS =========="
)


print(
"Average PSNR :",
swinir_results.PSNR_dB.mean()
)


print(
"Median PSNR :",
swinir_results.PSNR_dB.median()
)


print(
"Average SSIM :",
swinir_results.SSIM.mean()
)


print(
"Median SSIM :",
swinir_results.SSIM.median()
)


print(
"Average MSE :",
swinir_results.MSE.mean()
)


print(
"Average inference time :",
swinir_results.Inference_ms.mean(),
"ms/image"
)


print(
"Images tested:",
len(swinir_results)
)

========== SWINIR FINAL RESULTS ==========
Average PSNR : 27.523548899521337
Median PSNR : 27.53455113405181
Average SSIM : 0.7272044059563119
Median SSIM : 0.7612942078382168
Average MSE : 0.0027817846
Average inference time : 2.920389175415039 ms/image
Images tested: 320


In [24]:
RESTORE_DIR = "./results/swinir_sr/restored"


os.makedirs(
    RESTORE_DIR,
    exist_ok=True
)



model.eval()


with torch.no_grad():


    for idx,(lr,_) in enumerate(
        tqdm(test_loader)
    ):


        lr = lr.to(device)


        output = model(
            lr
        )


        output = torch.clamp(
            output,
            0,
            1
        )


        img = (
            output
            .cpu()
            .numpy()[0,0]
        )


        np.save(
            os.path.join(
                RESTORE_DIR,
                f"{idx:06d}.npy"
            ),
            img
        )


print(
"Restored images saved"
)

100%|██████████| 320/320 [00:02<00:00, 117.62it/s]

Restored images saved


In [25]:
worst_psnr = (
    swinir_results
    .sort_values(
        "PSNR_dB"
    )
    .head(20)
)


display(
    worst_psnr
)

,Filename,PSNR_dB,SSIM,MSE,Inference_ms
28,000627.npy,10.622701,0.256684,0.086642,2.810955
262,002041.npy,16.956356,0.389522,0.020154,2.808809
234,002317.npy,18.639777,0.462513,0.013678,2.853632
46,002393.npy,20.002627,0.658376,0.009994,2.855539
223,002264.npy,20.104726,0.827166,0.009762,2.886057
297,001215.npy,20.497059,0.716598,0.008919,3.152132
105,000641.npy,20.586194,0.265630,0.008737,2.839088
109,001636.npy,20.723911,0.768674,0.008465,2.839088
2,001415.npy,20.888544,0.694626,0.008150,2.831221
172,000206.npy,20.917098,0.679907,0.008096,2.852440


In [26]:
print(
"========== PERFORMANCE DISTRIBUTION =========="
)


display(
    swinir_results[
        [
            "PSNR_dB",
            "SSIM",
            "MSE"
        ]
    ].describe()
)

========== PERFORMANCE DISTRIBUTION ==========


,PSNR_dB,SSIM,MSE
count,320.000000,320.000000,320.000000
mean,27.523549,0.727204,0.002782
std,3.936332,0.145951,0.005245
min,10.622701,0.256684,0.000188
25%,24.571997,0.667117,0.000945
50%,27.534551,0.761294,0.001764
75%,30.246501,0.833881,0.003490
max,37.257312,0.945574,0.086642


In [27]:
comparison_df = pd.DataFrame(
[
{
"Method":"Bicubic",
"PSNR_dB":22.478416,
"SSIM":0.511662,
"MSE":0.007338
},

{
"Method":"SRCNN",
"PSNR_dB":27.529432,
"SSIM":0.725071,
"MSE":0.002734
},

{
"Method":"DnCNN",
"PSNR_dB":27.893001,
"SSIM":0.724960,
"MSE":0.002575
},

{
"Method":"EDSR",
"PSNR_dB":27.986647,
"SSIM":0.742506,
"MSE":0.002600
},

{
"Method":"NAF-SR",
"PSNR_dB":27.840080,
"SSIM":0.743259,
"MSE":0.002665
},

{
"Method":"Residual U-Net",
"PSNR_dB":28.776597,
"SSIM":0.772058,
"MSE":0.002261
},

{
"Method":"DA Residual U-Net",
"PSNR_dB":28.771419,
"SSIM":0.772428,
"MSE":0.002266
},

{
"Method":"SwinIR",
"PSNR_dB":
swinir_results.PSNR_dB.mean(),

"SSIM":
swinir_results.SSIM.mean(),

"MSE":
swinir_results.MSE.mean()
}

])


display(
comparison_df
)

,Method,PSNR_dB,SSIM,MSE
0,Bicubic,22.478416,0.511662,0.007338
1,SRCNN,27.529432,0.725071,0.002734
2,DnCNN,27.893001,0.724960,0.002575
3,EDSR,27.986647,0.742506,0.002600
4,NAF-SR,27.840080,0.743259,0.002665
5,Residual U-Net,28.776597,0.772058,0.002261
6,DA Residual U-Net,28.771419,0.772428,0.002266
7,SwinIR,27.523549,0.727204,0.002782


In [28]:
comparison_df.to_csv(
    "./results/final_model_comparison.csv",
    index=False
)


swinir_results.to_csv(
    "./results/swinir_test_results.csv",
    index=False
)


print(
"Final results saved"
)

Final results saved
